<a href="https://colab.research.google.com/github/daffu081/Employee-Attrition-Analysis/blob/main/fill_locations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fill `accessibility_locations.json` from `accessibility_map.json`

`accessibility_map.json` is the detailed venue map data. It's hand-edited and contains small JSON mistakes (trailing commas, a missing comma between two objects). This notebook:

1. Loads the map file and auto-repairs common JSON syntax mistakes.
2. Aggregates counts and nearby halls/entrances per category.
3. Writes the result into `accessibility_locations.json`.

Category mapping:

| locations.json key | map.json list      |
|---------------------|---------------------|
| `car_park`           | `regular_parking`   |
| `taxi`                | `taxi_ranks`        |
| `bus`                 | `bus_parking`       |
| `fuel_pump`           | `fuel_pump`         |
| `special_car_park`   | `special_parking`   |

In [6]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving accessibility_map.json to accessibility_map.json
Saving accessibility_locations.json to accessibility_locations.json
User uploaded file "accessibility_map.json" with length 47094 bytes
User uploaded file "accessibility_locations.json" with length 624 bytes


In [9]:
import json
import re
from pathlib import Path

# Update these if your files live somewhere else
MAP_PATH = Path("/content/accessibility_map.json")
OUTPUT_PATH = Path("/content/accessibility_locations.json")

CATEGORY_SOURCE = {
    "car_park": "regular_parking",
    "taxi": "taxi_ranks",
    "bus": "bus_parking",
    "fuel_pump": "fuel_pump",
    "special_car_park": "special_parking",
}

## Step 1: repair and load the map file

In [10]:
def repair_json(text: str) -> str:
    """Fix common hand-editing mistakes seen in accessibility_map.json."""
    # 1. Remove trailing commas before a closing ] or }
    #    e.g.  ["a", "b",]   or   { "x": 1, }
    text = re.sub(r",(\s*[\]}])", r"\1", text)

    # 2. Insert a missing comma between two adjacent objects in an array,
    #    e.g.  }\n    {   ->   },\n    {
    text = re.sub(r"\}(\s+)\{", r"},\1{", text)

    return text


def load_map(path: Path) -> dict:
    raw = path.read_text(encoding="utf-8")
    fixed = repair_json(raw)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError as e:
        lines = fixed.splitlines()
        context = lines[e.lineno - 1] if 0 < e.lineno <= len(lines) else ""
        raise ValueError(
            f"Could not parse {path} even after auto-repair: "
            f"{e.msg} at line {e.lineno}, column {e.colno}\n  >> {context}"
        ) from e


map_data = load_map(MAP_PATH)
print(f"Loaded {len(map_data)} top-level keys from {MAP_PATH}")

Loaded 15 top-level keys from /content/accessibility_map.json


## Step 2: aggregate counts, halls, and entrances per category

In [11]:
def collect_unique(entries: list, field: str) -> list:
    """Flatten `field` (a list-valued key) across all entries, drop blanks,
    de-duplicate, and sort."""
    values = set()
    for entry in entries:
        for v in entry.get(field, []) or []:
            if isinstance(v, str) and v.strip():
                values.add(v.strip())
    return sorted(values)


def build_locations(map_data: dict) -> dict:
    result = {}
    for category, source_key in CATEGORY_SOURCE.items():
        entries = map_data.get(source_key, []) or []
        result[category] = {
            "total_count": len(entries),
            "nearest_hall": collect_unique(entries, "near_halls"),
            "nearest_entrance": collect_unique(entries, "near_entrances"),
        }
    return result


locations = build_locations(map_data)
locations

{'car_park': {'total_count': 17,
  'nearest_hall': ['1.1',
   '1.2',
   '10',
   '16',
   '17',
   '18',
   '19',
   '2.1',
   '2.2',
   '20',
   '21',
   '26',
   '27',
   '7.1',
   '7.2',
   '8',
   '9'],
  'nearest_entrance': ['Hall 18 Entrance',
   'Hall 21 Entrance',
   'Hall 25-26 Entrance',
   'Hall 9 Entrance',
   'Hall79 Entrance',
   'Hub27 Entrance',
   'acc_entrance_east',
   'acc_entrance_south',
   'acc_north_entrance']},
 'taxi': {'total_count': 6,
  'nearest_hall': ['18', '7.1', '7.2'],
  'nearest_entrance': ['acc_entrance_east',
   'acc_entrance_kleiner_stern',
   'acc_entrance_north',
   'acc_entrance_south']},
 'bus': {'total_count': 4,
  'nearest_hall': ['14', '15', '16', '17', '18', '19', '20', '21'],
  'nearest_entrance': ['acc_entrance_east', 'acc_north_entrance']},
 'fuel_pump': {'total_count': 3,
  'nearest_hall': ['10.1',
   '10.2',
   '16',
   '17',
   '18',
   '7.1',
   '7.2',
   '7.3',
   '8.1',
   '8.2',
   '9'],
  'nearest_entrance': []},
 'special_car_pa

## Step 3: write the filled-in locations file

In [12]:
OUTPUT_PATH.write_text(json.dumps(locations, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

total = sum(v["total_count"] for v in locations.values())
print(f"Wrote {OUTPUT_PATH} ({total} total entries across {len(locations)} categories).")

Wrote /content/accessibility_locations.json (38 total entries across 5 categories).


In [13]:
from google.colab import files

files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes / caveats

- `accessibility_map.json` uses inconsistent entrance IDs in a few places (e.g. `acc_north_entrance` vs `acc_entrance_north`). These are kept as-is rather than silently merged — worth cleaning up at the source if you want a single canonical ID per entrance.
- If a syntax error can't be auto-repaired, the `ValueError` raised in Step 1 will point to the exact line/column to fix by hand.